In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim

print("Iniciando a sessão do Spark e configurando acesso ao Data Lake...")

spark = SparkSession.builder \
    .appName("Transformacao_Silver_Adzuna") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://host.docker.internal:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin1234") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# Lendo o CSV bruto
caminho_bronze = "s3a://radar-sp/bronze/vagas/bronze_vagas.csv"
df_bronze = spark.read.csv(caminho_bronze, header=True, inferSchema=True)

print(f"Total de vagas brutas carregadas: {df_bronze.count()}")

Iniciando a sessão do Spark e configurando acesso ao Data Lake...
Total de vagas brutas carregadas: 18


In [3]:
print("Aplicando regras de limpeza da Camada Silver...")

# 1. Remoção de Duplicatas (mantendo apenas IDs únicos)
df_deduplicado = df_bronze.dropDuplicates(["adzuna_id"])

# 2. Remoção de Nulos Críticos (vagas sem descrição ou título)
df_limpo = df_deduplicado.dropna(subset=["descricao", "titulo"])

# 3. Padronização de Strings (tudo minúsculo e sem espaços sobrando)
df_silver = df_limpo.withColumn("titulo_padronizado", trim(lower(col("titulo")))) \
                    .withColumn("descricao_padronizada", trim(lower(col("descricao"))))

print(f"Total de vagas limpas prontas para análise: {df_silver.count()}")
df_silver.select("titulo_padronizado", "empresa").show(5, truncate=False)

Aplicando regras de limpeza da Camada Silver...
Total de vagas limpas prontas para análise: 18
+-------------------------------------------+------------------+
|titulo_padronizado                         |empresa           |
+-------------------------------------------+------------------+
|data engineer                              |IBM               |
|lead data engineer                         |Sur               |
|data engineer - data foundry engineer      |Tractian          |
|sr. data analytics engineer                |Credix            |
|kdb/q developer – time series data engineer|emagine - Portugal|
+-------------------------------------------+------------------+
only showing top 5 rows



In [4]:
# O formato Parquet é muito mais rápido e leve que o CSV
caminho_silver = "s3a://radar-sp/silver/vagas"

print("Salvando dados limpos no Data Lake...")

# Mode 'overwrite' garante que não teremos arquivos duplicados se rodarmos a célula de novo
df_silver.write.mode("overwrite").parquet(caminho_silver)

print("✅ Camada Silver concluída com sucesso! Arquivos salvos no MinIO.")

Salvando dados limpos no Data Lake...
✅ Camada Silver concluída com sucesso! Arquivos salvos no MinIO.
